# Rearrangement SLM delivery benchmark


In [ ]:
import time

import numpy as np

import pytweezer.phasemask as pm
from pytweezer.analysis import analysis as an
from pytweezer.coordinators.rearrangement import Rearrangement
from pytweezer.drivers.slm import SLM, SimulatedSLM
from pytweezer.arduino import ArduinoPulser

try:
    from pytweezer.cpp import sum_pixel_values_cpp
except Exception as exc:
    sum_pixel_values_cpp = None
    print(f'C++ pixel summing unavailable ({type(exc).__name__}); using numpy.')

USE_REAL_SLM = True

try:
    slm.close()
    print('released the SLM board held by a previous run')
except NameError:
    pass
except Exception:
    pass

if not USE_REAL_SLM:
    slm = SimulatedSLM()
    print('in-process SLM: SimulatedSLM (forced by USE_REAL_SLM)')
else:
    try:
        slm = SLM()
        print(f'in-process SLM: real hardware {slm.get_dimensions()} '
              f'{slm.get_temperature():.1f} degC')
    except Exception as exc:
        slm = SimulatedSLM()
        print('in-process SLM: SimulatedSLM  <-- real driver FAILED')
        print(f'  {type(exc).__name__}: {exc}')
        print('  "SDK did not construct" or "found 0 boards" means another process still')
        print('  owns the board: an old Jupyter kernel, a running device server, or')
        print('  SLMServer.py. Close it (or restart this kernel) and re-run.')

In [ ]:
phasemask_generator = pm.OptimisationBasedPhasemaskGeneratorGPU(
    wavelength_um=0.852,
    focal_length_mm=17.3,
    slm_pitch_um=17,
    slm_res=(1024, 1024),
    input_beam_waist_mm=16,
    fresnel_f_mm=1072,
    blaze_dx_dy_um=(48, -4),
    zernike_coeff_dict={5: 1.195, 6: 0.725, 7: 0.970, 8: 0.478, 9: -1.091,
                        10: 0.303, 11: 0.021, 12: 0.072, 13: 0.049},
)

In [ ]:

spacing_um = 5.0
angle_deg = 2.5

initial_target = phasemask_generator.generate_weighted_array(
    np.ones((16, 16)), spacing_um, init_phase_randomness=1.0, angle_deg=angle_deg)
_, initial_terms, _ = phasemask_generator.superposition_optimization(
    initial_target, max_iter=1000, damping=0.5, verbose=False)

final_target = phasemask_generator.generate_weighted_array(
    np.ones((10, 10)), spacing_um, init_phase_randomness=1.0, angle_deg=angle_deg)
_, final_terms, _ = phasemask_generator.superposition_optimization(
    final_target, max_iter=1000, damping=0.5, verbose=False)

print(f'{initial_terms[0].size} initial sites -> {final_terms[0].size} target sites')

In [13]:
# Synthetic occupancy
array_shape = (16, 16)
roi_px = 256
window_size = 3
loading_probability = 0.6
bright_counts, background_counts = 1500, 50
occupancy_threshold = 8 * background_counts * window_size ** 2

grid_start, grid_step = 30, 12
grid_positions = {
    (i, j): (grid_start + i * grid_step, grid_start + j * grid_step)
    for i in range(array_shape[0]) for j in range(array_shape[1])
}

def make_synthetic_frame(rng):
    occupied = rng.random(array_shape) < loading_probability
    frame = rng.poisson(background_counts, size=(roi_px, roi_px)).astype(np.uint16)
    for (i, j), (y, x) in grid_positions.items():
        if occupied[i, j]:
            frame[y - 1:y + 2, x - 1:x + 2] += np.uint16(bright_counts)
    return frame

def extract_occupancy(frame, use_cpp=True):
    img = an.morphological_tophat_high_pass(frame, feature_size=10)
    if use_cpp and sum_pixel_values_cpp is not None and img.dtype == np.uint16:
        pixel_sums = sum_pixel_values_cpp.sum_pixel_values(
            img, grid_positions, array_shape, window_size=window_size)
    else:
        pixel_sums = an.sum_pixel_values(
            img, grid_positions, array_shape, window_size=window_size)
    return np.fliplr(pixel_sums).flatten() > occupancy_threshold

rng = np.random.default_rng(1234)
d0 = 1.2
n_repeats = 10

n_sites = array_shape[0] * array_shape[1]
print(f'atoms loaded this shot: {extract_occupancy(make_synthetic_frame(rng)).sum()} / {n_sites}')

atoms loaded this shot: 152 / 256


In [ ]:

class _NullCamera:
    def is_connected(self):
        return True

coordinator = Rearrangement({'camera': _NullCamera(), 'slm': slm}, {})

def time_pipelined(profile):
    """Generate and upload concurrently: frames stream to the SLM as produced."""
    durations, frame_counts = [], []
    for _ in range(n_repeats):
        occupancy_mask = extract_occupancy(make_synthetic_frame(rng))
        start = time.perf_counter()
        frames = phasemask_generator.iter_rearrangement_sequence(
            initial_terms, final_terms, occupancy_mask,
            d0=d0, profile=profile, to_host=False,
        )
        n_frames = coordinator._play_sequence_pipelined(frames)
        durations.append(time.perf_counter() - start)
        frame_counts.append(n_frames)
    return float(np.mean(durations)), int(np.mean(frame_counts))

def time_bulk():
    """Generate the whole sequence, then play it - the non-pipelined path."""
    durations, frame_counts = [], []
    for _ in range(n_repeats):
        occupancy_mask = extract_occupancy(make_synthetic_frame(rng))
        start = time.perf_counter()
        sequence = phasemask_generator.generate_rearrangement_sequence(
            initial_terms, final_terms, occupancy_mask, d0=d0)
        slm.run_sequence(sequence, fps=0)  # fps=0 -> no artificial inter-frame sleep
        durations.append(time.perf_counter() - start)
        frame_counts.append(len(sequence))
    return float(np.mean(durations)), int(np.mean(frame_counts))

In [ ]:
#  Pipelined vs bulk writes, both minimum-jerk:
pipelined_s, pipelined_frames = time_pipelined('minimum_jerk')
print(f'Pipelined (min-jerk) : {pipelined_s * 1e3:8.2f} ms | {pipelined_frames} frames | {pipelined_s / pipelined_frames * 1e3:.3f} ms/frame')

bulk_s, bulk_frames = time_bulk()
print(f'Bulk then play       : {bulk_s * 1e3:8.2f} ms | {bulk_frames} frames | {bulk_s / bulk_frames * 1e3:.3f} ms/frame')

In [ ]:
# (D) Preload the whole sequence into on-board memory, then clock it out with the
# hardware trigger. Only possible in-process: the full array is far over the RPC cap.
occupancy_mask = extract_occupancy(make_synthetic_frame(rng))

start = time.perf_counter()
full_sequence = phasemask_generator.generate_rearrangement_sequence(
    initial_terms, final_terms, occupancy_mask, d0=d0)
generate_s = time.perf_counter() - start

start = time.perf_counter()
slm.preload_sequence(full_sequence)
preload_s = time.perf_counter() - start

n_frames = len(full_sequence)
print(f'Generate whole seq   : {generate_s * 1e3:8.2f} ms | {n_frames} frames')
print(f'Preload upload       : {preload_s * 1e3:8.2f} ms')